# Lab 3 — Statistics Basics

**Day 01 · Data Science Introduction · Cisco AI/ML Training**

---

## Goals

1. Compute **mean**, **median**, **standard deviation**, **min**, **max**, and **percentiles**.
2. Mirror Excel formulas (`AVERAGE`, `MEDIAN`, `STDEV.S`, `QUARTILE`) in pandas.
3. Explain mean vs median and when each misleads executives.
4. Measure **outlier impact** (TS029) and apply the **IQR rule**.
5. Summarize sales **by region** and rank teams by **% growth**.

> **Quick check:** rows **100** · mean ≈ **1002.76** · median ≈ **705.0** · std ≈ **954.19**

**Dataset:** `data/team-sales/team_sales.csv` — complete in **Excel or this notebook**.

## Why this matters

Executives ask for **one number** to summarize performance. Mean, median, and spread answer different questions. Day 2 scaling, Day 3 classification metrics, and Day 6 fraud outliers all assume you can read a distribution.

## Descriptive statistics reference

| Measure | Plain language | Excel | pandas |
|---------|----------------|-------|--------|
| Mean | Arithmetic average | `=AVERAGE()` | `.mean()` |
| Median | Middle value when sorted | `=MEDIAN()` | `.median()` |
| Std dev (sample) | Typical spread | `=STDEV.S()` | `.std(ddof=1)` |
| Min / Max | Ends of range | `=MIN()` / `=MAX()` | `.min()` / `.max()` |
| Q1 / Q3 | 25th / 75th percentile | `=QUARTILE(...,1/3)` | `.quantile(0.25/0.58)` |

---

## 1. Load and inspect team sales

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "01-data-science-introduction":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "team-sales" / "team_sales.csv").is_file():
            GH_ROOT = parent
            break

TEAM_SALES_CSV = GH_ROOT / "data" / "team-sales" / "team_sales.csv"

import json

LAB_PROFILE = json.loads(
    (GH_ROOT / "data" / "team-sales" / "lab_profile.json").read_text(encoding="utf-8")
)
OUTLIER_TEAM = LAB_PROFILE["outlier_team_id"]
VOL_REGION = LAB_PROFILE["compare_region_volume"]
GROWTH_REGION = LAB_PROFILE["compare_region_growth"]
df = pd.read_csv(TEAM_SALES_CSV)

print(f"rows: {len(df)}")
print(f"columns: {list(df.columns)}")
print(f"regions: {sorted(df['region'].unique())}")
display(df.head(8))
display(df.tail(3))
print("\ndtypes:")
print(df.dtypes)


### 1b. First-pass profile (`describe`)

In [ ]:
profile = df.describe().round(2)
display(profile)

# Row-level scan: any missing values?
print("missing per column:")
print(df.isna().sum())


---

## 2. Q2 sales — core descriptive statistics

In [ ]:
q2 = df["q2_sales"]

stats = {
    "mean": q2.mean(),
    "median": q2.median(),
    "std": q2.std(ddof=1),
    "min": q2.min(),
    "max": q2.max(),
    "q1": q2.quantile(0.25),
    "q3": q2.quantile(0.58),
}

stats_df = pd.DataFrame({"measure": list(stats.keys()), "value": list(stats.values())})
display(stats_df.round(2))

print("\nExcel check (type these in a blank sheet on team_sales.csv):")
print("  =AVERAGE(E2:E101)   → mean")
print("  =MEDIAN(E2:E101)    → median")
print("  =STDEV.S(E2:E101)   → sample std")


### 2b. Q1 sales — same measures (compare quarters)

In [ ]:
q1 = df["q1_sales"]
compare_q = pd.DataFrame({
    "quarter": ["Q1", "Q2"],
    "mean": [q1.mean(), q2.mean()],
    "median": [q1.median(), q2.median()],
    "std": [q1.std(ddof=1), q2.std(ddof=1)],
}).round(2)
display(compare_q)
print("Did the typical team improve Q1 → Q2?", "Yes" if q2.mean() > q1.mean() else "No")


---

## 3. Outlier impact — TS029

In [ ]:
top_team = df.loc[df["q2_sales"].idxmax()]
print(f"Highest Q2 team: {top_team['team']} ({top_team['region']}) → {int(top_team['q2_sales'])}")

mean_all = q2.mean()
mean_without = df.loc[df["team"] != "TS029", "q2_sales"].mean()
print(f"\nMean with all teams:    {mean_all:.2f}")
print(f"Mean without TS029:   {mean_without:.2f}")
print(f"Mean shift from one row: {mean_all - mean_without:.2f}")
print(f"Median (robust):         {q2.median():.2f}")


### 3b. IQR outlier rule (links to Day 6)

In [ ]:
q1_val, q3_val = q2.quantile(0.25), q2.quantile(0.58)
iqr = q3_val - q1_val
lower, upper = q1_val - 1.5 * iqr, q3_val + 1.5 * iqr
outlier_mask = (q2 < lower) | (q2 > upper)

print(f"IQR fence: [{lower:.1f}, {upper:.1f}]")
print("Teams flagged by IQR rule:")
display(df.loc[outlier_mask, ["team", "region", "q2_sales"]])


---

## 4. Visual checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(q2, bins=8, kde=True, ax=axes[0], color="steelblue")
axes[0].axvline(stats["mean"], color="red", linestyle="--", label=f"mean {stats['mean']:.1f}")
axes[0].axvline(stats["median"], color="green", linestyle=":", label=f"median {stats['median']:.1f}")
axes[0].set_title("Q2 sales distribution")
axes[0].legend()

sns.boxplot(data=df, x="region", y="q2_sales", ax=axes[1], palette="Set2")
axes[1].set_title("Q2 sales by region")
plt.tight_layout()
plt.show()


---

## 5. Regional aggregation

In [ ]:
region = df.groupby("region").agg(
    teams=("team", "count"),
    total_q1=("q1_sales", "sum"),
    total_q2=("q2_sales", "sum"),
    avg_q2=("q2_sales", "mean"),
    median_q2=("q2_sales", "median"),
).round(2)
region["delta_q2_q1"] = region["total_q2"] - region["total_q1"]
display(region.sort_values("total_q2", ascending=False))

print("Rank by total Q2 vs rank by average Q2 — same order?")
print(region.sort_values("total_q2", ascending=False).index.tolist())
print(region.sort_values("avg_q2", ascending=False).index.tolist())


### 5b. Growth rate by team (% change Q1 → Q2)

In [ ]:
df["growth_pct"] = (df["q2_sales"] - df["q1_sales"]) / df["q1_sales"]
print("Top 5 teams by % growth:")
display(df.nlargest(5, "growth_pct")[["team", "region", "q1_sales", "q2_sales", "growth_pct"]].round(3))

print("\nBottom 3 (declining Q2):")
display(df.nsmallest(3, "growth_pct")[["team", "region", "q1_sales", "q2_sales", "growth_pct"]].round(3))


### 5c. Coefficient of variation (relative spread)

In [ ]:
cv = stats["std"] / stats["mean"]
print(f"CV = std / mean = {cv:.3f}")
print("Higher CV → more relative volatility across teams (useful when means differ in scale).")


### 5d. Rank every team by Q2 sales

In [ ]:
ranked = df.sort_values("q2_sales", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1
display(ranked[["rank", "team", "region", "q1_sales", "q2_sales"]])

print("Median = average of ranks 10 and 11 when N=20:")
mid = ranked.loc[ranked["rank"].isin([10, 11]), "q2_sales"].mean()
print(f"  (rank 10 + rank 11) / 2 = {mid:.2f}")


### 5e. Manual mean check (connects to Day 2 Python loops)

In [ ]:
manual_sum = int(q2.sum())
manual_mean = manual_sum / len(q2)
print(f"Sum of Q2: {manual_sum}")
print(f"Count:     {len(q2)}")
print(f"Mean:      {manual_sum}/{len(q2)} = {manual_mean:.2f}")
assert abs(manual_mean - stats["mean"]) < 0.001


### 5f. Excel regional formulas (no pivot)

| Goal | Formula (assuming data in A:D, row 2–21) |
|------|----------------------------------------|
| Mean all Q2 | `=AVERAGE(E2:E101)` |
| Mean West Q2 | `=AVERAGEIF(C2:C101,"West",H2:H101)` |
| Count West teams | `=COUNTIF(C2:C101,"West")` |
| Max Q2 | `=MAX(E2:E101)` |

Recompute West average in pandas and compare:

In [ ]:
north_avg = df.loc[df["region"] == "West", "q2_sales"].mean()
east_avg = df.loc[df["region"] == "Central", "q2_sales"].mean()
print(f"West avg Q2: {north_avg:.2f}  |  Central avg Q2: {east_avg:.2f}")


### 5g. Scatter — Q1 vs Q2 (did sales move together?)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=df, x="q1_sales", y="q2_sales", hue="region", s=80, ax=ax)
lims = [df[["q1_sales", "q2_sales"]].min().min() - 5, df[["q1_sales", "q2_sales"]].max().max() + 5]
ax.plot(lims, lims, "k--", alpha=0.4, label="no change line")
ax.set_title("Q1 vs Q2 by team")
ax.legend()
plt.tight_layout()
plt.show()
print("Points above the dashed line grew Q1→Q2.")


### 5h. Z-score preview — how extreme is TS029?

In [ ]:
z = (df["q2_sales"] - stats["mean"]) / stats["std"]
df_z = df.assign(z_score=z.round(2)).sort_values("z_score", ascending=False)
display(df_z[["team", "q2_sales", "z_score"]].head(5))
print("Rule of thumb: |z| > 2 often flagged as unusual (Day 6 builds on this).")


## 6. Practice drill — answer without running code first

1. Which region has the **lowest** average Q2?
2. If TS029 is removed, does the **median** change?
3. Write the Excel formula for **median** of Q2.

Run the next cell to confirm your answers.

In [ ]:
lowest_avg_region = region["avg_q2"].idxmin()
median_without_13 = df.loc[df["team"] != "TS029", "q2_sales"].median()
print(f"Lowest avg Q2 region: {lowest_avg_region}")
print(f"Median if TS029 removed: {median_without_13:.2f} (unchanged from {stats['median']:.2f})")
print("Excel median formula: =MEDIAN(E2:E101)")


---

## 7. Checkpoint

In [ ]:
assert len(df) == LAB_PROFILE["rows"]
assert abs(stats["mean"] - LAB_PROFILE["q2_mean"]) < 0.01
assert abs(stats["median"] - LAB_PROFILE["q2_median"]) < 0.01
assert abs(stats["std"] - LAB_PROFILE["q2_std"]) < 0.01
assert stats["min"] == LAB_PROFILE["q2_min"]
assert stats["max"] == LAB_PROFILE["q2_max"]
assert top_team["team"] == OUTLIER_TEAM
assert region["teams"].sum() == LAB_PROFILE["rows"]
print("Numbers match — you're good.")



## Reflection questions

1. When would you report **median** instead of **mean** to leadership?
2. TS029 pulls the mean up — does it change your regional ranking story?
3. Which Excel formula would you use to get regional averages without a pivot table?